# Exploratory Data Analysis: Ames Housing Dataset

## 1. Introduction
**Objective:** Use exploratory data analysis (no machine learning) to understand which factors drive house sale prices.

**Dataset:** `train.csv` (Ames Housing Dataset — ~1,460 rows, 81 columns).

**Guiding Questions:**
1. How is `SalePrice` distributed? Is it skewed?
2. Which numerical features correlate most strongly with `SalePrice`?
3. Does overall quality (`OverallQual`) change price in a clear, ordered way?
4. Do some neighborhoods consistently cost more than others?
5. Are there outliers, and are they errors or real but unusual sales?
6. Which columns have missing values, and which of those are not truly missing?

## 2. Data Loading (Step 1)
First, we import the required libraries, set a clean plotting style, and load `train.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("train.csv")
print("Shape:", df.shape)
df.head()

Next, let's check column data types, non-null counts, and list all columns that have missing values.

In [ ]:
df.info()

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({"Missing Count": missing, "Missing %": missing_pct})
print(f"Columns with missing values: {len(missing_df)}")
missing_df

**What we learned:** The dataset has 1,460 rows and 81 columns. Several columns (like `PoolQC`, `MiscFeature`, `Alley`, `Fence`, `FireplaceQu`) have high missing percentages—in this dataset, `NA` often means "feature absent" rather than unknown.

## 3. Data Understanding (Steps 2 & 3)
### Step 2: Classify Variables
We split columns into numerical and categorical groups, convert `ExterQual` and `KitchenQual` into ordered categoricals, and build a quick summary table. *(Note: `OverallQual` is stored as an integer 1–10, but it is really ordinal.)*

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

ordinal_map = {
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"]
}

for col, order in ordinal_map.items():
    if col in df.columns:
        df[col] = pd.Categorical(df[col], categories=order, ordered=True)

summary_rows = []
for col in df.columns:
    summary_rows.append({
        "Column": col,
        "Type": str(df[col].dtype),
        "Unique Values": df[col].nunique(),
        "Example Values": df[col].dropna().unique()[:3].tolist()
    })

summary_table = pd.DataFrame(summary_rows)
summary_table.head(15)

### Step 3: Descriptive Statistics
Let's summarise centre and spread for numeric columns, and inspect `SalePrice`, `GrLivArea`, and `LotArea` in detail.

In [ ]:
df[numeric_cols].describe().T.head(15)

In [ ]:
for col in ["SalePrice", "GrLivArea", "LotArea"]:
    s = df[col].dropna()
    mean_val = s.mean()
    median_val = s.median()
    mode_val = s.mode().iloc[0]
    std_val = s.std()
    var_val = s.var()
    range_val = s.max() - s.min()
    iqr_val = s.quantile(0.75) - s.quantile(0.25)
    skew_val = s.skew()
    kurt_val = s.kurt()
    skew_dir = "right skewed" if mean_val > median_val else "left skewed"

    print(f"--- {col} ({skew_dir}) ---")
    print(f"Mean: {mean_val:,.2f} | Median: {median_val:,.2f} | Mode: {mode_val:,.2f}")
    print(f"Std: {std_val:,.2f} | Var: {var_val:,.2f} | Range: {range_val:,.2f} | IQR: {iqr_val:,.2f}")
    print(f"Skewness: {skew_val:.2f} | Kurtosis: {kurt_val:.2f}\n")

In [ ]:
for col in ["Neighborhood", "HouseStyle"]:
    print(f"--- {col} (Mode: {df[col].mode().iloc[0]}) ---")
    print(df[col].value_counts().head(10), "\n")

**What we learned:** For `SalePrice`, the mean is noticeably higher than the median, which tells us the distribution is **right-skewed** (a few expensive houses pull the average up).

## 4. Data Cleaning (Step 9)
Let's check for duplicates, inconsistent categorical labels, and handle missing values properly before plotting and analysis.

In [ ]:
# 1. Check duplicates
dup_count = df.duplicated().sum()
print("Duplicate rows:", dup_count)
if dup_count > 0:
    df = df.drop_duplicates()
    print("New shape:", df.shape)

# 2. Check categoricals for extra spaces or casing issues
for col in categorical_cols:
    if df[col].dtype == "object":
        df[col] = df[col].str.strip()

# 3. Replace NA with 'None' where NA means feature absent
absent_cols = ["PoolQC", "Alley", "Fence", "FireplaceQu", "GarageType",
               "GarageFinish", "GarageQual", "GarageCond", "MiscFeature",
               "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "MasVnrType"]
for col in absent_cols:
    if col in df.columns:
        df[col] = df[col].fillna("None")

# 4. Fill numeric missing values (LotFrontage by Neighborhood median, others by column median)
if "LotFrontage" in df.columns:
    df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
        lambda x: x.fillna(x.median())
    )

for col in df.select_dtypes(include=[np.number]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

# Fill any remaining object columns with mode
for col in df.select_dtypes(include=["object"]).columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].mode().iloc[0])

print("Remaining missing values:", df.isnull().sum().sum())
print("Impossible YearBuilt > YrSold:", (df["YearBuilt"] > df["YrSold"]).sum())

**What we learned:** There are no duplicate rows, and replacing `NA` with `"None"` for absent features + imputing `LotFrontage` by neighborhood median leaves 0 missing values.

## 5. Univariate Analysis (Step 4)
Let's visualise the distribution of `SalePrice` (raw, boxplot, and log-transformed) and see how it varies by `OverallQual` and `Neighborhood`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df["SalePrice"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("SalePrice Distribution")
axes[0].set_xlabel("SalePrice")
axes[0].set_ylabel("Count")

sns.boxplot(y=df["SalePrice"], ax=axes[1], color="lightcoral")
axes[1].set_title("SalePrice Boxplot")

log_price = np.log1p(df["SalePrice"])
sns.histplot(log_price, kde=True, ax=axes[2], color="seagreen")
axes[2].set_title("Log(SalePrice) Distribution")
axes[2].set_xlabel("Log(SalePrice)")

plt.tight_layout()
plt.show()

print(f"Skewness before log: {df['SalePrice'].skew():.2f}")
print(f"Skewness after log:  {log_price.skew():.2f}")

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(x="OverallQual", y="SalePrice", data=df)
plt.title("SalePrice by Overall Quality")
plt.show()

In [ ]:
order = df.groupby("Neighborhood")["SalePrice"].median().sort_values().index
plt.figure(figsize=(14, 6))
sns.boxplot(x="Neighborhood", y="SalePrice", data=df, order=order)
plt.xticks(rotation=90)
plt.title("SalePrice by Neighborhood (Sorted by Median)")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x="OverallQual", data=df, ax=axes[0])
axes[0].set_title("Count of OverallQual")

sns.countplot(x="HouseStyle", data=df, ax=axes[1])
axes[1].set_title("Count of HouseStyle")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

**What we learned:**
- `SalePrice` is strongly right-skewed, and a log transform makes it nearly normal.
- Higher `OverallQual` clearly corresponds to higher `SalePrice`, with wider spread at the top.
- Neighborhoods differ substantially in median price.

## 6. Bivariate and Multivariate Analysis (Steps 5, 6, 7)
### Step 5: Correlation and Covariance
Let's find the numerical features most strongly correlated with `SalePrice` and compare covariance vs correlation.

In [ ]:
num_df = df.select_dtypes(include=[np.number])
cov_matrix = num_df.cov()
corr_matrix = num_df.corr()

top_corr = corr_matrix["SalePrice"].abs().sort_values(ascending=False)
print("Top 10 features correlated with SalePrice:")
print(corr_matrix.loc[top_corr.index[:11], "SalePrice"])

print("\nTop 3 features — Covariance vs Correlation with SalePrice:")
for feat in top_corr.index[1:4]:
    print(f"{feat:12s} | Cov: {cov_matrix.loc[feat, 'SalePrice']:12,.2f} | Corr: {corr_matrix.loc[feat, 'SalePrice']:.3f}")

**Why correlation is easier to compare than covariance:** Covariance depends on the scale/units of each variable, whereas Pearson correlation is standardized between -1 and +1.

**Correlation is not causation:** Features like `GarageArea` and `GarageCars` are strongly correlated with each other (multicollinearity) because bigger garages hold more cars.

### Step 6: Scatter Plots and Pair Plots
Let's visualise relationships with `SalePrice` and spot unusual points.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x="GrLivArea", y="SalePrice", data=df, alpha=0.6)
outliers_4k = df[(df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000)]
plt.scatter(outliers_4k["GrLivArea"], outliers_4k["SalePrice"], color="red", s=80, label="GrLivArea > 4000 & Low Price")
plt.title("GrLivArea vs SalePrice")
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x="TotalBsmtSF", y="SalePrice", data=df, ax=axes[0], alpha=0.6)
axes[0].set_title("TotalBsmtSF vs SalePrice")

sns.scatterplot(x="YearBuilt", y="SalePrice", data=df, ax=axes[1], alpha=0.6)
axes[1].set_title("YearBuilt vs SalePrice")
plt.tight_layout()
plt.show()

In [ ]:
selected = ["SalePrice", "GrLivArea", "TotalBsmtSF", "GarageArea", "YearBuilt"]
sns.pairplot(df[selected], diag_kind="kde")
plt.show()

### Step 7: Correlation Heatmap
A single clean heatmap of the top 10 features most correlated with `SalePrice`.

In [ ]:
top10 = top_corr.index[:11]
corr_top = df[top10].corr()
mask = np.triu(np.ones_like(corr_top, dtype=bool))

plt.figure(figsize=(10, 8))
sns.heatmap(corr_top, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Top 10 Features Correlated with SalePrice")
plt.show()

## 7. Outlier Analysis (Step 8)
Let's detect outliers using both the IQR method and the Z-score method (`|z| > 3`), compare them, and decide whether to remove extreme anomalies.

In [ ]:
def iqr_outliers(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return series[(series < q1 - 1.5 * iqr) | (series > q3 + 1.5 * iqr)].index

def zscore_outliers(series, threshold=3):
    z = (series - series.mean()) / series.std()
    return series[z.abs() > threshold].index

for col in ["GrLivArea", "SalePrice"]:
    a = iqr_outliers(df[col])
    b = zscore_outliers(df[col])
    overlap = len(set(a).intersection(set(b)))
    print(f"{col:10s} | IQR: {len(a):3d} | Z-score: {len(b):3d} | Overlap: {overlap}")

In [ ]:
# Remove the 2 extreme GrLivArea > 4000 & low SalePrice points
df_clean = df[~((df["GrLivArea"] > 4000) & (df["SalePrice"] < 300000))].copy()
print("Original rows:", len(df), "| Cleaned rows:", len(df_clean))

**Why Z-score flags fewer points on skewed data:** On a right-skewed variable, extreme high values inflate both the mean and standard deviation, making `|z| > 3` harder to exceed than the IQR fence.

## 8. Final Dashboard (Step 10)
Combining our key visuals into one clean 2×2 dashboard and saving it as `eda_dashboard.png`.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: Mean SalePrice by YearBuilt
yr_mean = df_clean.groupby("YearBuilt")["SalePrice"].mean()
axes[0, 0].plot(yr_mean.index, yr_mean.values, color="steelblue", label="Average price")
axes[0, 0].set_title("Average SalePrice by Year Built")
axes[0, 0].set_xlabel("YearBuilt")
axes[0, 0].set_ylabel("Mean SalePrice")
axes[0, 0].legend()

# Panel 2: Top 10 Neighborhoods by Mean SalePrice
top_nb = df_clean.groupby("Neighborhood")["SalePrice"].mean().sort_values().tail(10)
axes[0, 1].barh(top_nb.index, top_nb.values, color="teal")
axes[0, 1].set_title("Top 10 Neighborhoods by Mean SalePrice")
axes[0, 1].set_xlabel("Mean SalePrice")

# Panel 3: SalePrice Distribution with Mean & Median
sns.histplot(df_clean["SalePrice"], kde=True, ax=axes[1, 0], color="steelblue")
mean_p = df_clean["SalePrice"].mean()
med_p = df_clean["SalePrice"].median()
axes[1, 0].axvline(mean_p, color="red", linestyle="--", label=f"Mean: ${mean_p:,.0f}")
axes[1, 0].axvline(med_p, color="green", linestyle="-", label=f"Median: ${med_p:,.0f}")
axes[1, 0].set_title("SalePrice Distribution")
axes[1, 0].legend()

# Panel 4: GrLivArea vs SalePrice coloured by OverallQual
sns.scatterplot(x="GrLivArea", y="SalePrice", hue="OverallQual", palette="viridis",
                data=df_clean, ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title("GrLivArea vs SalePrice by OverallQual")

fig.suptitle("House Price EDA Summary", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_dashboard.png", dpi=150)
plt.show()

## 9. Conclusions
1. **`SalePrice` is right-skewed** (mean > median); a log transform makes it close to normal.
2. **`OverallQual` and `GrLivArea`** are among the strongest correlates of `SalePrice`.
3. **Location matters:** Neighborhoods show clear differences in median sale prices.
4. **Outliers:** A couple of very large houses (`GrLivArea > 4000`) sold at unusually low prices and were removed in `df_clean`.
5. **Missing values:** Many `NA` entries represent "feature absent" (e.g., no pool or garage) and were filled with `"None"` rather than dropped.